In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [2]:
data = pd.read_excel("housing_data.xlsx")

In [3]:
data.columns = data.columns.str.strip().str.lower()

In [4]:
data.columns = data.iloc[0]
data = data[1:]
data.reset_index(drop=True, inplace=True)

In [5]:
cols = ["House_ID" , "Location" ,"Area_Sqft"	, "Num_Bedrooms" , "Num_Bathrooms" , "Distance_From_City_KM" , "House_Age_Years" , "Num_Floors"	, "Price_Lakhs"]

for col in cols:
    data.loc[:,col] = pd.to_numeric(data[col], errors='coerce')

In [6]:
data = data.rename(columns = {"House_ID" : "ID",
                              "Area_Sqft" : "Area",
                             "Num_Bedrooms" : "Bedrooms",
                             "Num_Bathrooms" : "Bathrooms",
                             'Distance_From_City_KM' : "Distance",
                             'House_Age_Years' : "Age",
                             "Num_Floors" : "Floors",
                             "Price_Lakhs" : "Price"})

In [7]:
data.loc[:,'Price'] = pd.to_numeric(data['Price'], errors='coerce')

In [8]:
X = data.drop(columns=['Price'])   # features
y = data['Price']                 # target

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.90, random_state=91)

In [10]:
num_cols = X.select_dtypes(include=['int64','float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

In [11]:
num_pipeline = Pipeline([('imputer', SimpleImputer(strategy='mean')),('scaler', StandardScaler())])

In [12]:
cat_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))])

In [13]:
preprocessor = ColumnTransformer([('num', num_pipeline, num_cols),('cat', cat_pipeline, cat_cols)])

In [14]:
pipeline = Pipeline([('preprocessing', preprocessor),('model', LinearRegression())])

In [15]:
pipeline.fit(X_train, y_train)

C:\Users\ayyan malik\anaconda3\Lib\site-packages\sklearn\impute\_base.py:635: UserWarning: Skipping features without any observed values: ['ID' 'Location']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index([], dtype='object', name=0)),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['ID', 'Location', 'Area', 'Bedrooms', 'Bathrooms', 'Distance', 'Age',
       'Floors'],
      dtype='object', name=0))])),
                ('model', LinearRegression())])

In [16]:
y_pred = pipeline.predict(X_test)

C:\Users\ayyan malik\anaconda3\Lib\site-packages\sklearn\impute\_base.py:635: UserWarning: Skipping features without any observed values: ['ID' 'Location']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(


In [17]:
r2 = r2_score(y_test, y_pred)
print("R2 Score:", r2)

R2 Score: -0.00899958710666282
